In [ ]:
"""
Classical Machine Learning Models for Fake News Detection
Trains KNN and Naive Bayes on pre-computed TF-IDF features
"""

import numpy as np
import os
from pathlib import Path
from joblib import load, dump
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

BASE_DIR = Path(os.getcwd())
while BASE_DIR.name != "nlp-fake-news-detector-transformers" and BASE_DIR.parent != BASE_DIR:
    BASE_DIR = BASE_DIR.parent

FEATURES_DIR = BASE_DIR / "data" / "saved_features"
MODELS_DIR = BASE_DIR / "results" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)


class ClassicalModels:
    """Train and evaluate KNN and Naive Bayes classifiers"""
    
    def __init__(self):
        self.X_train = load(FEATURES_DIR / "X_train_tfidf.joblib")
        self.X_test = load(FEATURES_DIR / "X_test_tfidf.joblib")
        self.y_train = load(FEATURES_DIR / "y_train.joblib")
        self.y_test = load(FEATURES_DIR / "y_test.joblib")
        
        self.knn_model = None
        self.nb_model = None
    
    def train_knn(self, n_neighbors: int = 5):
        """Train KNN classifier"""
        self.knn_model = KNeighborsClassifier(n_neighbors=n_neighbors)
        self.knn_model.fit(self.X_train, self.y_train)

        y_pred = self.knn_model.predict(self.X_test)
        self.evaluate(self.knn_model, "KNN")

        return y_pred
    
    def train_naive_bayes(self, alpha: float = 1.0):
        """Train Naive Bayes classifier"""
        self.nb_model = MultinomialNB(alpha=alpha)
        self.nb_model.fit(self.X_train, self.y_train)

        y_pred = self.nb_model.predict(self.X_test)
        self.evaluate(self.nb_model, "Naive Bayes")

        return y_pred
    
    def evaluate(self, model, model_name: str):
        """Evaluate model and print metrics"""
        y_pred = model.predict(self.X_test)
        
        accuracy = accuracy_score(self.y_test, y_pred)
        precision = precision_score(self.y_test, y_pred)
        recall = recall_score(self.y_test, y_pred)
        f1 = f1_score(self.y_test, y_pred)
        
        print(f"\n{model_name} Results:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Precision: {precision:.4f}")
        print(f"Recall: {recall:.4f}")
        print(f"F1-Score: {f1:.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(self.y_test, y_pred))
        print("\nClassification Report:")
        print(classification_report(self.y_test, y_pred))
    
    def save_model(self, model, filename: str):
        """Save trained model"""
        dump(model, MODELS_DIR / filename)
        print(f"Model saved as {filename}")